In [ ]:
!pip install -q "transformers>=4.50" accelerate pandas

In [ ]:
import os, re, time, unicodedata, zipfile, gc
import torch, pandas as pd
from PIL import Image
from google.colab import userdata
from huggingface_hub import login
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration

login(token=userdata.get("HF_TOKEN"))
# Correcting the path to the zip file and ensuring extraction if images are missing
if not os.path.exists("veriseti/ana/orijinal"): # Check for a specific representative image folder
    print("Veri seti görüntüleri bulunamadı, arşivden çıkarılıyor...")
    try:
        # Corrected: Use the correct filename 'veriseti_son.zip'
        with zipfile.ZipFile("/content/veriseti/veriseti_son.zip", 'r') as zip_ref:
            zip_ref.extractall("veriseti") # Extract contents of zip into the 'veriseti' directory
        print("Veri seti görüntüleri başarıyla çıkarıldı.")
    except FileNotFoundError:
        print("Hata: '/content/veriseti/veriseti_son.zip' dosyası bulunamadı. Lütfen dosyanın doğru yolda olduğundan emin olun.")
        # You might want to exit or handle this error more gracefully
    except zipfile.BadZipFile:
        print("Hata: '/content/veriseti/veriseti_son.zip' geçerli bir zip dosyası değil.")
else:
    print("Veri seti görüntüleri zaten mevcut.")

print("GPU:", torch.cuda.get_device_name(0), "| bellek:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

In [ ]:
IMAGE_SOURCE = "orijinal"          # "orijinal" ya da "letterbox"
GROUPS = ["ana"]                   # ["ana", "taninmis_marka", "kontrol"] ekleyebilirsin
ONLY_VERIFIED = False              # True: sadece manifest'te dogrulandi=x olan satirlar
MAX_NEW_TOKENS = 256

MODELS = {
    224: "google/paligemma2-3b-mix-224",
    448: "google/paligemma2-3b-mix-448",
}
PROMPTS = {
    "ocr":    "ocr",
    "tr_soru": "answer tr Görselde yazan tüm metni yaz.",
    "en_soru": "answer en What is all the text written in the image?",
}
DTYPE = torch.float16 if torch.cuda.get_device_capability(0)[0] < 8 else torch.bfloat16
print("dtype:", DTYPE)

manifest = pd.read_csv("veriseti/manifest.csv", encoding="utf-8-sig").fillna("")
manifest = manifest[manifest["grup"].isin(GROUPS)]
if ONLY_VERIFIED:
    manifest = manifest[manifest["dogrulandi"].str.lower() == "x"]
print(len(manifest), "gorsel")

def image_path(row, res):
    folder = "orijinal" if IMAGE_SOURCE == "orijinal" else str(res)
    # Modified: Added an extra 'veriseti/' to the path due to nested extraction
    return f"veriseti/veriseti/{row['grup']}/{folder}/{row['dosya']}"

In [ ]:
results = []
for res, model_id in MODELS.items():
    print(f"\n=== {model_id} ===")
    processor = PaliGemmaProcessor.from_pretrained(model_id)
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=DTYPE, device_map="auto").eval()
    for _, row in manifest.iterrows():
        current_image_path = image_path(row, res)
        if not os.path.exists(current_image_path):
            print(f"Uyarı: Görüntü dosyası bulunamadı: {current_image_path}. Bu görsel atlanıyor.")
            continue
        image = Image.open(current_image_path).convert("RGB")
        for pname, prompt in PROMPTS.items():
            inputs = processor(text=prompt, images=image, return_tensors="pt").to(DTYPE).to(model.device)
            n_in = inputs["input_ids"].shape[-1]
            t0 = time.time()
            with torch.inference_mode():
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            text = processor.decode(out[0][n_in:], skip_special_tokens=True)
            results.append(dict(cozunurluk=res, prompt=pname, dosya=row["dosya"], grup=row["grup"],
                                cikti=text, sure_sn=round(time.time()-t0, 1)))
        print(row["dosya"], "tamam")
        pd.DataFrame(results).to_csv("sonuclar.csv", index=False, encoding="utf-8-sig")
    del model, processor
    gc.collect(); torch.cuda.empty_cache()

res_df = pd.DataFrame(results)
res_df.to_csv("sonuclar.csv", index=False, encoding="utf-8-sig")
res_df.head(6)

In [ ]:
pd.set_option("display.max_colwidth", 200)
sample = res_df[res_df["dosya"].isin(manifest["dosya"].head(3))]
sample[["cozunurluk","prompt","dosya","cikti"]]

In [ ]:
TR_FOLD = str.maketrans("çğıöşüâî", "cgiosuai")

def tr_lower(s):
    return s.replace("İ", "i").replace("I", "ı").lower()

def norm(s):
    s = unicodedata.normalize("NFC", str(s))
    s = tr_lower(s)
    s = re.sub(r"[^\w]+", " ", s, flags=re.UNICODE).replace("_", " ")
    return re.sub(r"\s+", " ", s).strip()

def fold(s):
    return s.translate(TR_FOLD)

def approx_substring_dist(ref, text):
    # ref'in text icindeki en iyi (en az hatali) konumla duzenleme mesafesi
    m, n = len(ref), len(text)
    if m == 0: return 0
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        cur = [i] + [0] * n
        for j in range(1, n + 1):
            cost = 0 if ref[i-1] == text[j-1] else 1
            cur[j] = min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + cost)
        prev = cur
    return min(prev)

def score(res_df, manifest):
    gt = manifest.set_index("dosya")
    res_df = res_df[res_df["dosya"].isin(gt.index)]   # manifest'ten cikarilan gorselleri atla
    rows = []
    for _, r in res_df.iterrows():
        out_n = norm(r["cikti"]); out_f = fold(out_n)
        for kind in ("ana", "ikincil"):
            for seg in [s.strip() for s in str(gt.loc[r["dosya"], kind + "_metin"]).split("|") if s.strip()]:
                ref = norm(seg)
                if not ref: continue
                rows.append(dict(
                    cozunurluk=r["cozunurluk"], prompt=r["prompt"], dosya=r["dosya"], tur=kind, parca=seg,
                    tam_bulundu=ref in out_n,
                    aksansiz_bulundu=fold(ref) in out_f,
                    hata_orani=round(approx_substring_dist(ref, out_n) / len(ref), 3)))
    return pd.DataFrame(rows)

sk = score(res_df, manifest)
sk.to_csv("skorlar_parca.csv", index=False, encoding="utf-8-sig")
sk.head()

In [ ]:
ozet = (sk.groupby(["tur", "cozunurluk", "prompt"])
          .agg(parca_sayisi=("parca", "count"),
               tam_bulundu=("tam_bulundu", "mean"),
               aksansiz_bulundu=("aksansiz_bulundu", "mean"),
               ort_hata_orani=("hata_orani", "mean"))
          .round(3).reset_index())
ozet.to_csv("ozet.csv", index=False, encoding="utf-8-sig")
ozet

In [ ]:
ana = sk[(sk["tur"] == "ana") & (sk["prompt"] == "ocr")]
gorsel = ana.pivot_table(index="dosya", columns="cozunurluk", values="tam_bulundu", aggfunc="mean").round(2)
gorsel["fark_448_224"] = gorsel[448] - gorsel[224]
gorsel.sort_values("fark_448_224", ascending=False)

In [ ]:
aksan = sk[(sk["aksansiz_bulundu"]) & (~sk["tam_bulundu"])]
print(len(aksan), "parcada aksan/harf hatasi")
aksan[["cozunurluk","prompt","dosya","parca"]].sort_values("dosya")

In [ ]:
from google.colab import files
for f in ["sonuclar.csv", "skorlar_parca.csv", "ozet.csv"]:
    files.download(f)